In [74]:

import cv2
import matplotlib.pyplot as plt
import json
import numpy as np
import yaml
import os
from tqdm import tqdm
from scipy.spatial.distance import euclidean
import pandas as pd
import copy
from sklearn.preprocessing import StandardScaler
import math


In [61]:
config_path = os.path.join('config.yaml')

with open(config_path) as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)
    
path_keypoints_2d_1person = cfg['path_keypoints_2d_1person']
connections = cfg['connections']
useless_points = cfg['coco_head_points']
base_connestion = cfg['base_connestion']
angles = cfg['angles']

In [3]:

data_list = []

# Iteracja przez pliki w folderze
for filename in tqdm(os.listdir(path_keypoints_2d_1person), desc="Wczytywanie plików .info"):
    if filename.endswith('.info'):
        file_path = os.path.join(path_keypoints_2d_1person, filename)

        # Wczytanie pliku JSON i dodanie do listy
        with open(file_path, 'r', encoding='utf-8') as file:
            json_data = json.load(file)[0]
            json_data['filename'] = filename
        
        # Dodanie słownika do listy
        data_list.append(json_data)

# Utworzenie DataFrame z listy słowników
keypoints_df = pd.DataFrame(data_list)

# czy to pomaga w notebookach?
del data_list

Wczytywanie plików .info: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 55746/55746 [12:30<00:00, 74.31it/s]


In [4]:
def remove_Points(df, points):
    df['keypoints_clear'] = df['keypoints'].apply(lambda x: [point for i, point in enumerate(x) if i not in points])
    df['keypoint_scores_clear'] = df['keypoint_scores'].apply(lambda x: [score for i, score in enumerate(x) if i not in points])

    return df

In [5]:
#keypoints_df_clear = delete_Points(keypoints_df, useless_points)

In [6]:

def count_lenghts(keypoints, connections):
    lenghts = [euclidean(keypoints[point1], keypoints[point2]) for point1, point2 in connections]
    return lenghts

def add_lenghts(df, connections, useless_points=[]):
    connections = [(x, y) for x, y in connections if x not in useless_points and y not in useless_points]
    df['lengths'] = df['keypoints'].apply(lambda x: count_lenghts(x, connections))
    

In [7]:
add_lenghts(keypoints_df, connections, useless_points=useless_points)

In [46]:
def add_relative_lenghts(df, base_connestion):
    df['base_length'] = df['keypoints'].apply(lambda x: count_lenghts(x, [base_connestion])[0])
    # gdyby oba stawy w tym samym pixelu (w 1percent 236 razy)
    df['base_length'] = df['base_length'].apply(lambda x: 1 if x < 1 else x)
    df['relative_lengths'] = df.apply(lambda row: [length / row['base_length'] for length in row['lengths']], axis=1)

In [47]:
add_relative_lenghts(keypoints_df, base_connestion)

In [59]:
import pandas as pd

def change_value(list_to_change, index, value):
    list_to_change[index]= value

    return list_to_change

def std_index(kolumna, index): 
    elements = kolumna.apply(lambda x: x[index])
    elements_mean = elements.mean()
    elements_std = elements.std()
    elements = [(elem - elements_mean) / elements_std for elem in elements]
    return kolumna.apply(lambda x: change_value(x, index, elements[index]))
 
def std_lengths(df):
# Standaryzacja dla wszystkich indeksów
    lenghts_list_len = len(df['relative_lengths'][0])
    # Załóż, że wszystkie listy mają tę samą długość
    df['relative_lengths_std'] = df['relative_lengths'].apply(copy.deepcopy)
    for i in range(lenghts_list_len):
        
        df['relative_lengths_std'] = std_index(df['relative_lengths_std'], i)

# Wyświetlenie zmodyfikowanej ramki danych
std_lengths(keypoints_df)

In [70]:
def angle3pt(a, b, c):
    """Counterclockwise angle in degrees by turning from a to c around b
        Returns a float between 0.0 and 360.0"""
    ang = math.degrees(
        math.atan2(c[1]-b[1], c[0]-b[0]) - math.atan2(a[1]-b[1], a[0]-b[0]))
    return ang + 360 if ang < 0 else ang

def get_base_angle(mid_point, end_point):
    base_end_point = [0.0, mid_point[1]]
    return angle3pt(base_end_point, mid_point, end_point)

# dla base_points nie działa tak jak powinno
def get_angle3(a, b, c, base_points=None):
    if base_points:
        base_angle = get_base_angle(basepoint1, basepoint2)
    else:
        base_angle = 0.0
    angle = angle3pt(a, b, c) - base_angle
    return angle + 360 if angle < 0 else angle

In [72]:
def count_angles(keypoints, angles, base_points=None):
    angles_size = [get_angle3(keypoints[point1], keypoints[point2], keypoints[point3], base_points) for point1, point2, point3 in angles]
    return angles_size

def add_angles(df, angles, useless_points=[], base_connestion=None):
    angles = [(x, y, z) for x, y, z in angles if x not in useless_points and y not in useless_points]
    df['angles'] = df['keypoints'].apply(lambda x: count_angles(x, angles, base_connestion))

In [75]:
add_angles(keypoints_df, angles)

In [79]:
def count_variant_angle(keypoints, connections):
    lenghts = [get_base_angle(keypoints[point1], keypoints[point2]) for point1, point2 in connections]
    return lenghts

def add_variant_angle(df, connections, useless_points=[]):
    connections = [(x, y) for x, y in connections if x not in useless_points and y not in useless_points]
    df['variant_angles'] = df['keypoints'].apply(lambda x: count_variant_angle(x, connections))

In [80]:
add_variant_angle(keypoints_df, connections)